# Serializadores customizados

Até agora ficamos satisfeitos com a forma como o Pydantic serializa os valores dos campos.

Mas às vezes, especialmente com certos tipos de dados como datetimes, queremos controlar como os campos são serializados.

Um exemplo típico é definir como um `date` ou `datetime` é serializado.

Outro é padronizar o número de casas decimais de floats.

Seja qual for a necessidade, dá para controlar a serialização dos campos facilmente.

Vamos usar um **decorador** do Pydantic, `@field_serializer`, que controla a serialização no nível do campo.

In [1]:
from pydantic import BaseModel, field_serializer

O decorador tem argumentos que definem a qual campo o serializador se aplica e como ele é aplicado.

Uma opção importante:
- `when_used`: por padrão o serializador customizado é sempre usado, mas há outras opções:
    - `always`: o padrão; o serializador roda ao serializar tanto para dict quanto para JSON
    - `unless-none`: o serializador não é usado se o valor for `None`
    - `json`: o serializador só é usado ao serializar para JSON
    - `json-unless-none`: usado ao serializar para JSON, exceto se o valor for `None`

Há também a opção de modo `plain` × `wrap`, raramente usada, que não é coberta no curso.

Vejamos a opção `when_used` e em que circunstâncias o serializador é chamado.

In [2]:
from datetime import datetime

class Model(BaseModel):
    dt: datetime | None = None

    @field_serializer("dt", when_used="always")
    def serialize_name(self, value):
        print(f"type = {type(value)}")
        return value

In [3]:
m = Model(dt="2020-01-01T12:00:00")
print(f"m: {m!r}")

m: Model(dt=datetime.datetime(2020, 1, 1, 12, 0))


A primeira coisa a perceber: o serializador roda depois que o modelo foi populado — então `value` já tem o tipo válido (ou `None`,
neste exemplo, já que o campo é anulável).

In [4]:
print(f"m.model_dump(): {m.model_dump()!r}")

type = <class 'datetime.datetime'>
m.model_dump(): {'dt': datetime.datetime(2020, 1, 1, 12, 0)}


O serializador foi chamado — e o valor devolvido é o que aparece na saída serializada.

Serializando para JSON:

In [5]:
print(f"m.model_dump_json(): {m.model_dump_json()!r}")

type = <class 'datetime.datetime'>
m.model_dump_json(): '{"dt":"2020-01-01T12:00:00"}'


O dado foi serializado corretamente, porque o Pydantic sabe serializar datetime para JSON (usa o padrão ISO).

Porém, ao serializar para JSON talvez não queiramos essa representação.

Voltamos a isso em seguida.

Vejamos o que acontece se `dt` for `None`:

In [6]:
m = Model()
print(f"m: {m!r}")

m: Model(dt=None)


In [7]:
print(f"m.model_dump(): {m.model_dump()!r}")

type = <class 'NoneType'>
m.model_dump(): {'dt': None}


In [8]:
print(f"m.model_dump_json(): {m.model_dump_json()!r}")

type = <class 'NoneType'>
m.model_dump_json(): '{"dt":null}'


O serializador foi chamado nos dois casos.

Para não rodar o serializador quando o valor for `None`, usamos outra opção de `when_used`:

In [9]:
from datetime import datetime

class Model(BaseModel):
    dt: datetime | None = None

    @field_serializer("dt", when_used="unless-none")
    def serialize_name(self, value):
        print(f"type = {type(value)}")
        return value

In [10]:
m = Model(dt="2020-01-01T12:00:00")
print(f"m: {m!r}")

m: Model(dt=datetime.datetime(2020, 1, 1, 12, 0))


In [11]:
print(f"m.model_dump(): {m.model_dump()!r}")

type = <class 'datetime.datetime'>
m.model_dump(): {'dt': datetime.datetime(2020, 1, 1, 12, 0)}


In [12]:
m = Model()
print(f"m: {m!r}")

m: Model(dt=None)


In [13]:
print(f"m.model_dump(): {m.model_dump()!r}")

m.model_dump(): {'dt': None}


In [14]:
print(f"m.model_dump_json(): {m.model_dump_json()!r}")

m.model_dump_json(): '{"dt":null}'


O serializador não foi chamado quando `dt` era `None`.

Voltando ao caso em que só queremos mudar a serialização para JSON. Podemos aceitar a serialização para dicionário, mas, na saída JSON,
queremos o datetime formatado assim:

```
2020/1/1 12:00 PM
```

Usamos o método `strftime()`:

In [15]:
dt = datetime(2020, 1, 1, 12, 0, 0)
print(f"dt.isoformat(): {dt.isoformat()!r}")

dt.isoformat(): '2020-01-01T12:00:00'


In [16]:
print(f'dt.strftime("%Y/%m/%d %I:%M %p"): {dt.strftime("%Y/%m/%d %I:%M %p")!r}')

dt.strftime("%Y/%m/%d %I:%M %p"): '2020/01/01 12:00 PM'


Usando isso no serializador, configurado para valer só na serialização JSON e não quando o valor for `None`:

In [17]:
from datetime import datetime

class Model(BaseModel):
    dt: datetime | None = None

    @field_serializer("dt", when_used="json-unless-none")
    def serialize_name(self, value):
        print(f"type = {type(value)}")
        return value.strftime("%Y/%m/%d %I:%M %p")

In [18]:
m = Model(dt="2020-01-01T12:00:00")
print(f"m: {m!r}")

m: Model(dt=datetime.datetime(2020, 1, 1, 12, 0))


In [19]:
print(f"m.model_dump(): {m.model_dump()!r}")

m.model_dump(): {'dt': datetime.datetime(2020, 1, 1, 12, 0)}


Serializar para dicionário não rodou o serializador.

Já na serialização para JSON:

In [20]:
print(f"m.model_dump_json(): {m.model_dump_json()!r}")

type = <class 'datetime.datetime'>
m.model_dump_json(): '{"dt":"2020/01/01 12:00 PM"}'


E, pela configuração, o serializador não é chamado se o valor for `None`:

In [21]:
m = Model()
print(f"m: {m!r}")

m: Model(dt=None)


In [22]:
print(f"m.model_dump_json(): {m.model_dump_json()!r}")

m.model_dump_json(): '{"dt":null}'


Agora suponha que queremos serializações diferentes para dicionário e para JSON.

Precisamos saber, dentro do serializador, qual serialização está sendo feita, e reagir de acordo.

O Pydantic oferece mais um argumento para a função serializadora — do tipo `FieldSerializationInfo`. Vejamos:

In [23]:
from pydantic import FieldSerializationInfo

In [24]:
class Model(BaseModel):
    dt: datetime | None = None

    @field_serializer("dt", when_used="unless-none")
    def dt_serializer(self, value, info: FieldSerializationInfo):
        print(f"info={info}")
        return value

In [25]:
m = Model(dt=datetime(2020, 1, 1))
print(f"m: {m!r}")

m: Model(dt=datetime.datetime(2020, 1, 1, 0, 0))


In [26]:
print(f"m.model_dump(): {m.model_dump()!r}")

info=SerializationInfo(include=None, exclude=None, context=None, mode='python', by_alias=False, exclude_unset=False, exclude_defaults=False, exclude_none=False, exclude_computed_fields=False, round_trip=False, serialize_as_any=False)
m.model_dump(): {'dt': datetime.datetime(2020, 1, 1, 0, 0)}


Repare no valor de `mode` no objeto `info`: `python`.

Agora, dump para JSON:

In [27]:
print(f"m.model_dump_json(): {m.model_dump_json()!r}")

info=SerializationInfo(include=None, exclude=None, context=None, mode='json', by_alias=False, exclude_unset=False, exclude_defaults=False, exclude_none=False, exclude_computed_fields=False, round_trip=False, serialize_as_any=False)
m.model_dump_json(): '{"dt":"2020-01-01T00:00:00"}'


Agora `mode` é `json`.

Poderíamos usar isso, mas `FieldSerializationInfo` oferece o método `mode_is_json()`, que é mais direto.

In [28]:
class Model(BaseModel):
    dt: datetime | None = None

    @field_serializer("dt", when_used="unless-none")
    def dt_serializer(self, value, info: FieldSerializationInfo):
        print(f"mode_is_json={info.mode_is_json()}")
        return value

In [29]:
m = Model(dt=datetime(2020, 1, 1))

In [30]:
print(f"m.model_dump(): {m.model_dump()!r}")

mode_is_json=False
m.model_dump(): {'dt': datetime.datetime(2020, 1, 1, 0, 0)}


In [31]:
print(f"m.model_dump_json(): {m.model_dump_json()!r}")

mode_is_json=True
m.model_dump_json(): '{"dt":"2020-01-01T00:00:00"}'


Vejamos uma situação em que essa flexibilidade é útil.

Digamos que o serializador deva garantir que datetimes sejam sempre serializados como horários UTC com fuso (*timezone aware*). Além disso,
queremos a notação `Z` para UTC, em vez do `+00:00` que o `isoformat()` do Python costuma devolver.

Dá para escrever isso facilmente em Python usando a biblioteca `pytz`.

Para este exemplo, garanta que `pytz` está instalado no ambiente virtual.

Vamos escrever uma função Python que, dado um datetime:
- se ele for *naive* (sem fuso), torna-o *aware*, assumindo que já estava em UTC
- se for *aware*, converte para UTC

In [32]:
import pytz

def make_utc(dt: datetime) -> datetime:
    if dt.tzinfo is None:
        dt = pytz.utc.localize(dt)
    else:
        dt = dt.astimezone(pytz.utc)
    return dt

Usando:

In [33]:
dt = make_utc(datetime(2026, 11, 11, 9, 30))
print(f"dt: {dt!r}")

dt: datetime.datetime(2026, 11, 11, 9, 30, tzinfo=<UTC>)


In [34]:
print(f"dt.isoformat(): {dt.isoformat()!r}")

dt.isoformat(): '2026-11-11T09:30:00+00:00'


Precisamos mudar o formato serializado; como sabemos que estará sempre em UTC, é simples:

In [35]:
print(f'dt.strftime("%Y-%m-%dT%H:%M:%SZ"): {dt.strftime("%Y-%m-%dT%H:%M:%SZ")!r}')

dt.strftime("%Y-%m-%dT%H:%M:%SZ"): '2026-11-11T09:30:00Z'


Uma função para isso:

In [36]:
def dt_utc_json_serializer(dt: datetime) -> str:
    dt = make_utc(dt)
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")

E agora implementando no serializador customizado:

In [37]:
class Model(BaseModel):
    dt: datetime | None = None

    @field_serializer("dt", when_used="unless-none")
    def dt_serializer(self, dt, info: FieldSerializationInfo):
        if info.mode_is_json():
            return dt_utc_json_serializer(dt)
        return make_utc(dt)

In [38]:
m = Model(dt=datetime(2020, 1, 1))
print(f"m: {m!r}")

m: Model(dt=datetime.datetime(2020, 1, 1, 0, 0))


In [39]:
print(f"m.model_dump(): {m.model_dump()!r}")

m.model_dump(): {'dt': datetime.datetime(2020, 1, 1, 0, 0, tzinfo=<UTC>)}


In [40]:
print(f"m.model_dump_json(): {m.model_dump_json()!r}")

m.model_dump_json(): '{"dt":"2020-01-01T00:00:00Z"}'


E se tivermos um datetime *aware* que ainda não está em UTC:

In [41]:
eastern = pytz.timezone('US/Eastern')
dt = eastern.localize(datetime(2020, 1, 1))
print(f"dt: {dt!r}")

dt: datetime.datetime(2020, 1, 1, 0, 0, tzinfo=<DstTzInfo 'US/Eastern' EST-1 day, 19:00:00 STD>)


Usando no modelo:

In [42]:
m = Model(dt=dt)
print(f"m: {m!r}")

m: Model(dt=datetime.datetime(2020, 1, 1, 0, 0, tzinfo=<DstTzInfo 'US/Eastern' EST-1 day, 19:00:00 STD>))


In [43]:
print(f"m.model_dump(): {m.model_dump()!r}")

m.model_dump(): {'dt': datetime.datetime(2020, 1, 1, 5, 0, tzinfo=<UTC>)}


In [44]:
print(f"m.model_dump_json(): {m.model_dump_json()!r}")

m.model_dump_json(): '{"dt":"2020-01-01T05:00:00Z"}'


> 📝 **Nota de revisão:** desde o Python 3.9 a biblioteca padrão tem `zoneinfo` + `datetime.timezone.utc`, que dispensam o `pytz`
> (e sua pegadinha de `localize`). A mesma função, sem dependência externa:

In [45]:
from datetime import timezone
from zoneinfo import ZoneInfo

def make_utc_stdlib(dt: datetime) -> datetime:
    return dt.replace(tzinfo=timezone.utc) if dt.tzinfo is None else dt.astimezone(timezone.utc)

print(f'make_utc_stdlib(datetime(2020, 1, 1, tzinfo=ZoneInfo("US/Eastern"))) == make_utc(eastern.localize(datetime(2020, 1, 1))): {make_utc_stdlib(datetime(2020, 1, 1, tzinfo=ZoneInfo("US/Eastern"))) == make_utc(eastern.localize(datetime(2020, 1, 1)))!r}')

make_utc_stdlib(datetime(2020, 1, 1, tzinfo=ZoneInfo("US/Eastern"))) == make_utc(eastern.localize(datetime(2020, 1, 1))): True
